# Advanced usage: write a model, tune it through the simulator

*PowerUp 2026, advanced usage (about 15 minutes, presenter-driven)*

Notebook 01 used the simulator as a black box: files in, trajectories and
eigenvalues out. This notebook uses two design decisions that let you change what
is inside:

1. **Models are composed, not forked.** A converter is a filter, an angle source, a
   voltage controller, an inner controller and optionally a PLL, each a small class
   selected by name in `sim_param.txt`. Machines work the same way with AVR, governor,
   PSS and shaft strategies. A new behavior is a new class, registered with one call.
2. **The whole system is one symbolic expression.** With `parametric=True` the
   simulator re-assembles the same equations with every device parameter as a CasADi
   symbol, so the gradient of anything you compute from a trajectory, with respect to
   any parameter, is exact.

Plan: (A) write a converter control strategy that is not in the package and register
it, (B) tune the gains of a shipped strategy by gradient descent through the DAE
solver, (C) ask the s-plane which knob moves which mode.

In [ ]:
import importlib.metadata, importlib.util, os, subprocess, sys

if importlib.util.find_spec("hermess") is None:                      # Colab
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "hermess>=1.7.1"])
    # Colab preimports numpy; if pip just upgraded it, the kernel holds a stale
    # mix of old and new files. A one-time restart fixes it; the install sticks.
    stale = [m for m in ("numpy", "pandas", "matplotlib") if m in sys.modules
             and importlib.metadata.version(m) != sys.modules[m].__version__]
    if stale:
        print("pip upgraded", ", ".join(stale), "under the running kernel;")
        print("restarting the runtime now. When it reconnects (a few seconds),")
        print("run the cells again from the top; the install is already done.")
        os.kill(os.getpid(), 9)

In [ ]:
%matplotlib inline
import inspect, shutil, time
import numpy as np
import pandas as pd
import casadi as ca
import matplotlib.pyplot as plt
import hermess
from hermess.analysis import *

print("hermess", hermess.__version__)
assert tuple(int(x) for x in hermess.__version__.split(".")[:3]) >= (1, 7, 1), \
    "this notebook needs hermess 1.7.1 or newer: pip install -U hermess"

In [ ]:
# Same conventions as notebook 01: the package's analysis layer above, and this
# cell holds the two notebook-side pieces, the ETH style and the run settings.
ETH = {"blue": "#215CAF", "petrol": "#007894", "green": "#627313", "bronze": "#8E6713",
       "red": "#B7352D", "purple": "#A7117A", "grey": "#6F6F6F"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[ETH[k] for k in
        ("blue", "petrol", "red", "green", "bronze", "purple", "grey")]),
    "font.family": "serif",
    "font.serif": ["Latin Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm", "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 110, "legend.frameon": False,
})

WS = dict(line_dyn=True, ts=0.001, incl_lim=False, quiet=True,
          int_scheme_sim_options={"reltol": 1e-8, "abstol": 1e-10,
                                  "max_num_steps": 100000, "jit": False})

def run(system, **overrides):
    """simulate() with the workshop settings pre-filled; any Config field overrides."""
    return simulate(system, **{**WS, **overrides})

## Part A: a control strategy the package does not have

What can be selected from a system file today:

The grid-forming converter in notebook 01 sets its frequency with a power droop
(`angle = "Droop"`). That behavior lives in one small class:

An angle source declares its states, parameters and setpoints, writes its differential
equations into `dae.f` inside `fgcall`, and says how to initialize itself at steady
state. The host converter registers whatever the strategy declares and reads the
parameter values from `sim_param.txt`.

The registry above also lists `VSM`, a virtual synchronous machine. That class was
added to the package recently through the workflow shown here, and it is validated
against a PSCAD EMT trace. The module docstring keeps a wish list,
and the next entry on it is **dVOC**, dispatchable virtual oscillator control
(Colombino et al.). Its angle behavior is

$$\omega_c = \omega_{net} + \frac{\eta}{\hat v^2}\,\big(p^\star - p_c\big), \qquad
  \dot\delta_c = \omega_b\,(\omega_c - \omega_{ref}),$$

the oscillator's phase dynamics, with the amplitude branch held at its setpoint
$\hat v = V_{ref}$ (the full dVOC also steers the voltage magnitude; that half would be
a `VoltageControl` strategy, and is the rest of the pull request). Two things differ
from the droop: the power is the **unfiltered** converter power $p_c$ (the oscillator
itself does the smoothing, there is no measurement-filter state), and the gain is
normalized by the voltage magnitude.

`hermess.register` makes the class selectable by name, like a shipped
strategy. It works as a decorator (registering under the class name) or as a call
(with an alias), the same function takes user-defined devices, AVRs, governors,
PSSs and shafts, and `unregister` removes a name again.

In [ ]:
@register                     # selectable as  angle = "DVOCAngle"
class DVOCAngle(AngleSource):
    """dVOC angle dynamics: oscillator phase synchronization on the instantaneous
    power error, normalized by the voltage setpoint. Amplitude branch not included."""

    def states(self):      return ["delta_c"]
    def units(self):       return ["rad"]
    def params(self):      return {"eta": 0.02}
    def x0(self):          return {"delta_c": 0.0}
    def setpoints(self):   return {"Pref": 0.5}
    def descriptions(self):
        return {"eta": "dVOC synchronization gain",
                "delta_c": "converter-frame angle relative to the network",
                "Pref": "active power set point"}

    def fgcall(self, host, dae, omega_ref_vec, omega_b):
        # the phase law (network frequency + eta * power error / Vref**2),
        # publish it as host.omega_c, write the ODE into dae.f, return omega_c
        raise NotImplementedError("typed live")

    def finit_sequential(self, host, dae, Pc, delta_c):
        # synchronized steady state: omega_c = omega_net pins Pref = Pc
        raise NotImplementedError("typed live")

register(DVOCAngle, "dVOC")   # ... and under the name the literature uses
registered("angle")

Now select it. We copy the 3-bus system three times: the shipped droop, the shipped
VSM, and our dVOC, and use a larger load step (+30 MW) so the responses are easy to
tell apart. The dVOC gain is set to $\eta = K_p \hat v^2$ so its steady-state droop
matches the droop converter. For the VSM we pick a large virtual inertia $T_a$ and
little damping $K_d$ on purpose, so that its dynamics are visible; Part B tunes
those two gains.

In [ ]:
SYS = "3bus_loadstep"
STEP = 'Disturbance, time = 1.0, type = "LOAD", bus = "2", p_delta = 30, q_delta = 0'

root = copy_system(SYS)
set_disturbances(root, SYS, [STEP])

for variant in ("3bus_vsm", "3bus_dvoc"):
    shutil.copytree(root / SYS, root / variant, dirs_exist_ok=True)

# the shipped VSM needs a PLL strategy for its damping term (numbers from the
# PSCAD-validated case); Kw is the inverse steady-state droop, Kd the damping
set_param(root, "3bus_vsm", "GFMI2", angle='"VSM"', Ta=32.0, Kd=2.0, Kw=20.0,
          pll='"Kaura"', Kpll_p=0.084, Kpll_i=4.69, omega_lp=500)
set_param(root, "3bus_dvoc", "GFMI2", angle='"dVOC"', eta=0.0110)

print(f"the three systems live in '{root}'")
show_system(root, "3bus_dvoc", which=("sim_param.txt",))

Three controls, one line changed in the system file each time. The droop and the dVOC
agree in steady state by construction; the dVOC acts on the unfiltered power, so its
first response is faster. The VSM's frequency $\omega_{vsm}$ is a state and cannot
jump, the large virtual inertia lets the first swing go deeper, and a lightly
damped oscillation appears in the VSM's frequency and power.

`plot_states` is the small-multiples view of a device, every state on its own
axes, and it works for the strategy written above as for a shipped one: one angle state, `delta_c`, next to the filter and inner-loop states
of the host converter.

The small-signal analysis names that behavior: a mode in which the VSM's speed and
angle swing against the synchronous machine's rotor, with a low damping ratio.
Part B works on that mode.

**From notebook cell to contribution.** The pattern is the same for AVR, governor,
PSS and shaft strategies, for the converter's filter, voltage, inner and PLL slots,
and for whole devices (a class in the running session is addressed by its class name
in the first column of `sim_param.txt`). The frequency `omega_c` the class publishes
appears in the results container and in the GUI's signal tree without further
registration. From this cell to a pull request, the steps are the ones the shipped
VSM went through: move the class into
`hermess/devices/inverter_angle.py`, add its docstring with the equations and symbol
table (that page of the docs is generated from it), add a small validation system,
and open the PR. `CONTRIBUTING.md` and the "adding your own model" section of the
docs spell out the three steps.

## Part B: tune the controller through the simulator

### B1. Every parameter, as a symbol

The VSM above is underdamped. Instead of sweeping its gains, we compute the gradient
of an objective with respect to them. `parametric=True` re-runs the same build with every float
device parameter lifted into a CasADi symbol; the run itself is unchanged (same
trajectories, same eigenvalues), but the symbolic equations are kept on the side as
`dae.parametric_model`. Setpoints, line and grid parameters stay numeric (they enter
through the admittance matrices; that is phase 2 of the differentiability plan),
and the operating point is not differentiated, so the sensitivities are exact for
parameters the power flow does not see, which control gains are.

For the tuning we drop the load step and study the ring-down instead: the system
starts with the synchronous machine 0.1 percent fast (the state right after any active-power
imbalance) and relaxes back. The quasi-static network keeps the model small (the
LCL warning from notebook 01 is acceptable here; B5 validates the tuned gains on the
full model anyway), and a disturbance-free run never rebuilds the equations mid-run,
so the stashed parametric model describes the whole horizon.

In [ ]:
pd.DataFrame([(e.device._type, e.name, e.values.size, e.values.round(4))
              for e in model.entries],
             columns=["device", "parameter", "n", "value"]).tail(8)

### B2. The objective rides along as a quadrature

`model.dae_dict()` is a CasADi integrator dictionary of the same equations with the
parameter vector appended, so a CasADi `integrator` turns the model into a
differentiable map from parameters to the trajectory. The objective is written as one
more equation of the same DAE, a quadrature the solver integrates alongside the
states (which also keeps the reverse sweep well conditioned). The converter's own
electrical power `gfm.Pc` is published as a symbolic expression in the same state
vector, so the VSM's effort term needs no manual derivation:

$$J(p) = \int_0^T \big[\Delta f_{SG}^2 + \Delta f_{VSM}^2
  + \rho\,\Delta P_c^2\big]\,dt.$$

In [ ]:
T_H = 4.0
i_sgw, i_om = int(sg.omega[0]), int(gfm.omega_vsm[0])
fn, Sn, RHO = dae_p.fn, float(gfm.Sn[0]), 1e-4
Pc0 = float(ca.Function("Pc", [model.x], [gfm.Pc])(dae_p.xinit))

d = model.dae_dict()                              # x, z, p, ode, alg of the model
d["quad"] = (fn * (model.x[i_sgw] - 1))**2 \
    + (fn * (model.x[i_om] - 1))**2 \
    + RHO * (Sn * (gfm.Pc - Pc0))**2              # the published power, symbolically

J_int = ca.integrator("J_int", "idas", d, 0.0, [T_H], {"reltol": 1e-8, "abstol": 1e-8})

x0 = np.array(dae_p.xinit)
x0[sg.omega] += 2e-3                              # the synchronous machine 0.1 % fast

p_mx = ca.MX.sym("p", model.p.numel())
out = J_int(x0=x0, z0=dae_p.yinit, p=ca.vertcat(ca.DM.ones(dae_p.nx), p_mx))
J = out["qf"][0, -1]

J_fun    = ca.Function("J",  [p_mx], [J])
grad_fun = ca.Function("dJ", [p_mx], [ca.gradient(J, p_mx)])

### B3. The exact gradient, checked once against finite differences

One call gives $\partial J/\partial p$ for **all** lifted parameters at once (CasADi
resolves this in reverse mode, so the cost does not grow with their number).

In [ ]:
t0 = time.perf_counter(); Jv = float(J_fun(model.p_val)); dt_J = time.perf_counter() - t0
t0 = time.perf_counter(); g = np.array(grad_fun(model.p_val)).ravel(); dt_g = time.perf_counter() - t0

for idx, label in ((i_Ta, "Ta"), (i_Kd, "Kd")):
    h = 0.02 * abs(model.p_val[idx])
    pp, pm = model.p_val.copy(), model.p_val.copy()
    pp[idx] += h; pm[idx] -= h
    fd = (float(J_fun(pp)) - float(J_fun(pm))) / (2 * h)
    print(f"dJ/d{label}: exact {g[idx]: .6e}   finite differences {fd: .6e}")
print(f"\nJ = {Jv:.5f}; one trajectory {dt_J:.2f} s, trajectory + full gradient {dt_g:.2f} s "
      f"({model.p.numel()} parameters)")

Which parameters matter most for this objective? Sort by the relative sensitivity
$|p\,\partial J/\partial p|$ (the change of $J$ per relative change of the
parameter). The gradient covers every lifted parameter, including the ones we did
not plan to look at. (Ratings and reactances also
shift the operating point, which the parametric build holds fixed; for control
gains, which leave the power flow untouched, the numbers are exact.)

In [ ]:
labels = np.concatenate([[f"{list(e.device.int)[u]}:{e.name}" for u in range(e.values.size)]
                         for e in model.entries])
sens = pd.Series(np.abs(g * model.p_val), index=labels).sort_values(ascending=False)
sens.head(8).round(5)

### B4. Descend

We move only the two VSM gains $T_a$ and $K_d$, with the simplest possible
optimizer, a normalized gradient step in $(\log T_a, \log K_d)$. A coarse grid of
plain evaluations shows the objective around the path.

In [ ]:
Tas, Kds = [4, 8, 16, 32, 48], [2, 5, 12, 30, 75, 180]
t0 = time.perf_counter()
Jgrid = np.zeros((len(Tas), len(Kds)))
for a, ta in enumerate(Tas):
    for b, kd in enumerate(Kds):
        pv = model.p_val.copy(); pv[i_Ta], pv[i_Kd] = ta, kd
        Jgrid[a, b] = float(J_fun(pv))
print(f"landscape: {Jgrid.size} simulations in {time.perf_counter() - t0:.1f} s")

In [ ]:
STEP_SIZE, N_STEPS = 0.18, 10
pv = model.p_val.copy()
path = []
t0 = time.perf_counter()
for k in range(N_STEPS):
    Jk = float(J_fun(pv))
    gk = np.array(grad_fun(pv)).ravel()
    path.append((pv[i_Ta], pv[i_Kd], Jk))
    step = np.array([pv[i_Ta] * gk[i_Ta], pv[i_Kd] * gk[i_Kd]])   # dJ/dlog p
    step = STEP_SIZE * step / np.linalg.norm(step)
    pv[i_Ta] *= np.exp(-step[0]); pv[i_Kd] *= np.exp(-step[1])
Ta_star, Kd_star = pv[i_Ta], pv[i_Kd]
print(f"{N_STEPS} gradient steps in {time.perf_counter() - t0:.1f} s")
for k, (ta, kd, Jk) in enumerate(path):
    print(f"  step {k:2d}: Ta = {ta:5.2f}  Kd = {kd:6.1f}  J = {Jk:.5f}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
TT, KK = np.meshgrid(Tas, Kds, indexing="ij")
cs = axs[0].contourf(KK, TT, Jgrid, levels=14, cmap="Blues_r")
fig.colorbar(cs, ax=axs[0], label="J")
pt, pk, _ = zip(*path)
axs[0].plot(pk, pt, "o-", color=ETH["red"], ms=4, label="gradient descent")
axs[0].plot(pk[0], pt[0], "s", color=ETH["red"], ms=8)
axs[0].plot(pk[-1], pt[-1], "*", color=ETH["red"], ms=14)
axs[0].set(xscale="log", yscale="log", xlabel="$K_d$", ylabel="$T_a$ [s]",
           title="Objective landscape"); axs[0].legend(loc="upper left")

p_star = model.p_val.copy(); p_star[i_Ta], p_star[i_Kd] = Ta_star, Kd_star
tgrid = np.arange(0.02, T_H + 1e-9, 0.02)
I_plot = ca.integrator("I", "idas", model.dae_dict(), 0.0, tgrid,
                       {"reltol": 1e-8, "abstol": 1e-8})
X0 = np.asarray(I_plot(x0=x0, z0=dae_p.yinit, p=np.concatenate([np.ones(dae_p.nx), model.p_val]))["xf"])
Xs = np.asarray(I_plot(x0=x0, z0=dae_p.yinit, p=np.concatenate([np.ones(dae_p.nx), p_star]))["xf"])
for X, lbl, c in [(X0, "as built ($T_a$=32, $K_d$=2)", ETH["grey"]),
                  (Xs, f"tuned ($T_a$={Ta_star:.1f}, $K_d$={Kd_star:.0f})", ETH["red"])]:
    axs[1].plot(tgrid, fn * X[i_sgw, :], ls="--", color=c, label=f"machine, {lbl}")
    axs[1].plot(tgrid, fn * X[i_om, :], color=c, label=f"converter, {lbl}")
axs[1].set(xlabel="time [s]", ylabel="frequency [Hz]", title="Ring-down, before and after")
axs[1].legend(fontsize=8)
fig.tight_layout()

Two observations. First, the descent reduces the virtual inertia $T_a$ and barely
touches $K_d$: the damping gain has little effect here, because the fast PLL tracks
the VSM's own speed and leaves the $K_d (\omega_{vsm} - \omega_{pll})$ term almost
no slip to act on. Second, a ring-down objective has no use for inertia, which only
stores swing energy; the reason to keep $T_a$ is something this objective does not
include, the rate of change of frequency right after a step. The result depends on
the objective, which is a modeling choice.

### B5. Put the answer back into the model

The optimizer worked on the symbolic side. Writing the gains back into
`sim_param.txt` and re-running the original load step (dynamic network, full
nonlinear model) closes the loop on the full model.

The eigenvalues show the effect. The VSM mode, the least damped slow mode as built
at 0.88 Hz with a damping ratio of 0.13, sits near 1.7 Hz with a damping ratio of
0.24 after tuning and is still the least damped slow mode. The synchronous
machine's exciter mode, which the objective does not include, lost a little damping
(0.28 to 0.25).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
plot_modes(dae_vsm,   ax=ax, label="as built", fmax=3, xlim=(-12, 0.5))
plot_modes(dae_tuned, ax=ax, label="tuned", color=ETH["red"], fmax=3, xlim=(-12, 0.5))

slow = dict(n=3, oscillatory_only=True, max_freq=3)      # the electromechanical band
display(modal_table(dae_vsm, **slow), modal_table(dae_tuned, **slow))

Ten gradient steps replaced a parameter sweep, and each gradient covered every
lifted parameter at the cost of about one extra simulation. That cost does not grow
with the number of parameters, so the same loop applies with a neural network in
place of $(T_a, K_d)$. `to_csv(dae_tuned, ...)` exports the columns a figure needs,
as in notebook 01.

## Part C: which knob moves which mode

Small-signal analysis says *where* the modes are. The parametric model also says
*where they go*: with $A(p)$ symbolic, the derivative of any eigenvalue with respect
to any parameter is

$$\frac{\partial \lambda_k}{\partial p_j}
  = \frac{w_k^{H}\,\frac{\partial A}{\partial p_j}\,v_k}{w_k^{H} v_k},$$

with $v_k, w_k$ the right and left eigenvectors, evaluated from the same parametric
equations. Without parameter derivatives this would be a finite-difference study
per parameter.

We assemble the reduced state matrix $A(p) = f_x - f_y\,g_y^{-1} g_x$ from the
parametric equations, check it against the one the simulator used, and draw, for the
slow modes, the arrow $p\,\partial\lambda/\partial p$ for the parameters with the
largest effect: the first-order shift of the mode per relative change of that
parameter. (The operating
point is held fixed, which is exact for the control gains shown here; parameters
that shift the power flow would add an initial-condition term.)

In [ ]:
r = model.rhs()
ones_s = ca.DM.ones(r.s.numel())
f_s = ca.substitute(r.f, r.s, ones_s)          # switches closed, as in the run
g_s = ca.substitute(r.g, r.s, ones_s)

fx, fy = ca.jacobian(f_s, r.x), ca.jacobian(f_s, r.y)
gx, gy = ca.jacobian(g_s, r.x), ca.jacobian(g_s, r.y)
A_expr = fx - ca.mtimes(fy, ca.solve(gy, gx))

A_fun  = ca.Function("A",  [r.x, r.y, r.p], [A_expr])
dA_fun = ca.Function("dA", [r.x, r.y, r.p], [ca.jacobian(ca.reshape(A_expr, -1, 1), r.p)])

A0 = np.asarray(A_fun(dae_p.xinit, dae_p.yinit, model.p_val))
dae_p.eigenvalue_analysis()
print("matches the simulator's state matrix:", np.allclose(A0, np.asarray(dae_p.A), atol=1e-8))

In [ ]:
import scipy.linalg

lam, W, V = scipy.linalg.eig(A0, left=True)
dA = np.asarray(dA_fun(dae_p.xinit, dae_p.yinit, model.p_val))   # (n*n) x n_p
n = A0.shape[0]

# d(lambda_k)/d(p_j) for every mode and every parameter, in one einsum
dAj = dA.reshape(n, n, -1, order="F")
num = np.einsum("ik,ijp,jk->kp", W.conj(), dAj, V)
dlam = num / np.einsum("ik,ik->k", W.conj(), V)[:, None]

In [ ]:
GAINS = {"Ta", "Kd", "Kw", "Kp", "Kq", "KA", "Rd", "Kpv", "Kiv", "Kpc", "Kic",
         "Kpll_p", "Kpll_i", "H", "D"}
keep = np.array([lab.split(":")[1] in GAINS for lab in labels])

slow = [k for k in range(n) if 0 < lam[k].imag / (2 * np.pi) < 3]
fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.scatter(lam.real, lam.imag / (2 * np.pi), s=18, color=ETH["grey"], alpha=0.4)
colors = [ETH[c] for c in ("blue", "red", "petrol", "green", "bronze", "purple")]
seen = {}
for k in slow:
    move = model.p_val * dlam[k, :]                  # the shift per relative turn
    rank = np.argsort(-np.abs(move))
    rank = [j for j in rank if keep[j]][:4]
    ax.plot(lam[k].real, lam[k].imag / (2 * np.pi), "o", color=ETH["blue"], ms=7)
    for j in rank:
        c = seen.setdefault(labels[j], colors[len(seen) % len(colors)])
        ax.annotate("", xy=(lam[k].real + move[j].real, (lam[k].imag + move[j].imag) / (2 * np.pi)),
                    xytext=(lam[k].real, lam[k].imag / (2 * np.pi)),
                    arrowprops=dict(arrowstyle="-|>", color=c, lw=1.6))
for lab, c in seen.items():
    ax.plot([], [], color=c, label=lab)
ax.axvline(0, color=ETH["grey"], lw=0.8)
ax.set(xlabel="Re $\\lambda$ [1/s]", ylabel="frequency [Hz]",
       xlim=(-2.4, 0.4), ylim=(-0.05, 1.15),
       title="Slow modes, and where each knob pushes them "
             "(arrow = $p\\,\\partial\\lambda/\\partial p$)")
ax.legend(fontsize=9, title="parameter")
fig.tight_layout()

This guides the tuning through the modes: it answers
"what do I tune" before any re-simulation, for every controller in the system at
once, from the same parametric model Part B used. For this system, each slow mode
responds to a different subsystem. The VSM mode responds to the VSM's inertia $T_a$
(destabilizing when raised) and its frequency droop $K_w$, while $K_d$ has little
effect on it, consistent with the trajectory gradient in Part B. The exciter mode
follows the AVR gain, and the governor mode the synchronous machine's droop and
inertia.

## Where this goes

* **Scale.** The tutorial uses a small example. The IEEE 39-bus system with two
  grid-forming and three grid-following converters (306 states) simulates 5 s in a
  few seconds, and its full eigenvalue analysis takes a tenth of a second; set
  `RUN_39BUS = True` below to try it.
* **Your models.** A strategy is a class; a device is a class; a system is two files;
  a contribution is a pull request. Finish the dVOC completely and send me a pull request!

In [ ]:
RUN_39BUS = False
if RUN_39BUS:
    t0 = time.perf_counter()
    dae39 = run("ieee39_conv", T_end=5.0)
    print(f"39-bus with 2 GFM + 3 GFL: nx = {dae39.nx}, {time.perf_counter() - t0:.1f} s")
    summary(dae39)
    plot(dae39, ["*:f", "bus*:v"])
    display(metrics(dae39).round(4))

## Recap

* New behavior is a small class, registered with one call; the host does the
  bookkeeping.
  The symbolic outputs your class publishes surface in the results container and
  the GUI without further work.
* `parametric=True` makes every device parameter a symbol: exact gradients of
  trajectories and eigenvalues, at a cost that does not grow with the number of
  parameters.
* The tuned result goes back into the system file, so the next run uses it.